# 📓 Notebook 01 — YouTube Transcript Pipeline

**Project:** IncidentIQ — AI-powered Incident Intelligence for Firefighters  
**Goal of this notebook:** Build the data ingestion pipeline that transforms a YouTube video into a searchable knowledge base stored in ChromaDB.

## What this notebook covers
1. Install and import all required packages
2. Extract the transcript from a YouTube video using `youtube-transcript-api`
3. Split the transcript into chunks using LangChain's text splitter
4. Generate vector embeddings using OpenAI's embedding model
5. Store the embeddings in a local ChromaDB vector database
6. Run a first similarity search to verify everything works

## Why this matters
This pipeline is the **foundation of the entire RAG system**. Without a properly chunked and embedded transcript, the agent cannot answer questions accurately. Every other notebook depends on this one.

---

## 1. Install required packages
Install all dependencies needed for this notebook.  
Run this cell once — you can skip it on subsequent runs.

In [ ]:
!pip install youtube-transcript-api langchain langchain-openai langchain-community chromadb python-dotenv -q

## 2. Import libraries and load environment variables
We use `python-dotenv` to load the OpenAI API key from a `.env` file.  
Never hardcode API keys directly in your code — always use environment variables.

In [ ]:
import os
from dotenv import load_dotenv

# YouTube transcript extraction
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

# LangChain components
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# ChromaDB
import chromadb

# Load environment variables from .env file
load_dotenv()

# Verify OpenAI API key is loaded
assert os.getenv("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not found. Check your .env file."
print("✅ Environment variables loaded successfully.")

## 3. Define the YouTube URL
Set the YouTube URL you want to process.  
The pipeline extracts the video ID automatically from any standard YouTube URL format.  

**Test video:** Karel Lambert — *Murphy comes to Brussels* (High Rise Fire Case Study)  
This is a real firefighting incident presentation — perfect for our use case.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
YOUTUBE_URL = "https://www.youtube.com/watch?v=7OH5FEWWM_k"
CHROMA_PATH = "../chroma_db"          # Persistent storage path for ChromaDB
COLLECTION_NAME = "incidentiq"        # Name of the ChromaDB collection
CHUNK_SIZE = 500                       # Max characters per chunk
CHUNK_OVERLAP = 50                     # Overlap between chunks to preserve context
EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI embedding model — fast and cost-efficient


def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Supports standard (watch?v=) and shortened (youtu.be/) formats.
    """
    if "v=" in url:
        return url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]
    else:
        raise ValueError(f"Could not extract video ID from URL: {url}")


video_id = extract_video_id(YOUTUBE_URL)
print(f"✅ Video ID extracted: {video_id}")

## 4. Fetch the YouTube transcript
We use `youtube-transcript-api` to fetch the official subtitles from YouTube.  
This approach is **safe and fast** — it only downloads text, never audio or video.  

**Language priority:** English first, then Dutch, then French.  
If no transcript is available, the user receives a clear error message — no Whisper fallback needed for our use case.

In [ ]:
def fetch_transcript(video_id: str, languages: list = ["en", "nl", "fr"]) -> tuple[str, str]:
    """
    Fetch the transcript of a YouTube video.
    Returns a tuple of (plain_text, timestamped_text).
    Timestamped text preserves [MM:SS] markers for accurate source citation.
    """
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
    except NoTranscriptFound:
        raise ValueError(
            f"No transcript found for video {video_id}. "
            "Please use a video with subtitles enabled (CC button visible in YouTube)."
        )
    except TranscriptsDisabled:
        raise ValueError(
            f"Transcripts are disabled for video {video_id}. "
            "The video owner has turned off subtitles."
        )

    # Build plain text — used for overview stats
    plain_text = " ".join([entry["text"] for entry in transcript_list])

    # Build timestamped text — preserves timing for accurate answers like "this was said at 4:32"
    timestamped_parts = []
    for entry in transcript_list:
        mins = int(entry["start"] // 60)
        secs = int(entry["start"] % 60)
        timestamped_parts.append(f"[{mins:02d}:{secs:02d}] {entry['text']}")
    timestamped_text = " ".join(timestamped_parts)

    return plain_text, timestamped_text


plain_text, timestamped_text = fetch_transcript(video_id)

print(f"✅ Transcript fetched successfully.")
print(f"   Total characters (plain):       {len(plain_text):,}")
print(f"   Total characters (timestamped): {len(timestamped_text):,}")
print(f"\n📄 First 500 characters preview:")
print("-" * 60)
print(plain_text[:500])

## 5. Split the transcript into chunks
Large texts cannot be stored as a single vector — we need to split them into smaller chunks.  

**Why chunking matters:**
- Embeddings work best on short, focused pieces of text
- Smaller chunks = more precise retrieval
- Overlap between chunks ensures context is not lost at boundaries

We use `RecursiveCharacterTextSplitter` which splits on natural boundaries (paragraphs → sentences → words) before splitting mid-word.

In [ ]:
def split_transcript(text: str, video_id: str, source_url: str) -> list[Document]:
    """
    Split transcript text into overlapping chunks and wrap them as LangChain Documents.
    Each Document carries metadata (video_id, source URL) for later filtering and citation.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " "],  # Split on natural boundaries first
    )

    chunks = splitter.create_documents(
        texts=[text],
        metadatas=[{
            "video_id": video_id,
            "source": source_url,
            "chunk_size": CHUNK_SIZE,
        }]
    )
    return chunks


chunks = split_transcript(timestamped_text, video_id, YOUTUBE_URL)

print(f"✅ Transcript split into {len(chunks)} chunks.")
print(f"   Chunk size:    {CHUNK_SIZE} characters")
print(f"   Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"\n📦 Example chunk (chunk 3):")
print("-" * 60)
print(chunks[2].page_content)
print(f"\n🏷️  Metadata: {chunks[2].metadata}")

## 6. Initialize ChromaDB and the embedding model
ChromaDB is our local vector database — it stores text chunks as high-dimensional vectors.  

**Why ChromaDB?**
- Runs entirely locally — no external account or API needed
- Persists data between sessions via `PersistentClient`
- Free and open-source

**Why `text-embedding-3-small`?**
- OpenAI's most cost-efficient embedding model
- 1536 dimensions — high quality for semantic search
- Fast: embeds thousands of chunks in seconds

In [ ]:
# Initialize the OpenAI embedding model
# This model converts text into numerical vectors that capture semantic meaning
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Initialize ChromaDB with persistent storage
# PersistentClient saves the database to disk so data survives between sessions
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

print(f"✅ ChromaDB initialized at: {CHROMA_PATH}")
print(f"✅ Embedding model ready:   {EMBEDDING_MODEL}")

## 7. Store chunks in ChromaDB
We embed each chunk using the OpenAI model and store the resulting vectors in ChromaDB.  

**What happens under the hood:**
1. Each chunk of text is sent to the OpenAI embedding API
2. The API returns a vector of 1536 numbers representing the semantic meaning
3. ChromaDB stores both the vector and the original text for later retrieval

This process runs **once per video** — subsequent questions use the stored vectors, not the API.

In [ ]:
def store_in_chromadb(
    chunks: list[Document],
    embedding_model: OpenAIEmbeddings,
    collection_name: str,
    chroma_path: str,
) -> Chroma:
    """
    Embed all chunks and store them in a ChromaDB collection.
    If the collection already contains this video, existing chunks are cleared first
    to avoid duplicate entries.
    Returns the Chroma vectorstore instance for immediate use.
    """
    vectorstore = Chroma(
        client=chromadb.PersistentClient(path=chroma_path),
        collection_name=collection_name,
        embedding_function=embedding_model,
    )

    # Check if this video is already stored — avoid duplicates
    existing = vectorstore.get(where={"video_id": chunks[0].metadata["video_id"]})
    if existing["ids"]:
        print(f"⚠️  Found {len(existing['ids'])} existing chunks for this video. Clearing...")
        vectorstore.delete(ids=existing["ids"])

    # Embed and store all chunks
    # LangChain handles batching automatically to avoid API rate limits
    vectorstore.add_documents(chunks)

    return vectorstore


print(f"⏳ Embedding {len(chunks)} chunks and storing in ChromaDB...")
print(f"   This may take 10-30 seconds depending on transcript length.")

vectorstore = store_in_chromadb(chunks, embedding_model, COLLECTION_NAME, CHROMA_PATH)

print(f"\n✅ All chunks stored successfully in ChromaDB.")
print(f"   Collection: '{COLLECTION_NAME}'")
print(f"   Total chunks stored: {len(chunks)}")

## 8. Verify: Run a first similarity search
We test the pipeline by running a semantic search query against the stored vectors.  

**How similarity search works:**
1. The query is embedded using the same OpenAI model
2. ChromaDB computes the cosine similarity between the query vector and all stored vectors
3. The top-k most similar chunks are returned

This is the exact mechanism the RAG agent uses to answer questions.

In [ ]:
# Test query — directly related to the Karel Lambert video content
TEST_QUERY = "What mistakes were made during the high rise fire incident in Brussels?"

# Retrieve top 3 most relevant chunks
results = vectorstore.similarity_search(TEST_QUERY, k=3)

print(f"🔍 Query: '{TEST_QUERY}'")
print(f"\n📋 Top {len(results)} most relevant chunks retrieved:")
print("=" * 60)

for i, doc in enumerate(results):
    print(f"\n[Chunk {i+1}]")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:300]}...")
    print("-" * 40)

## 9. Pipeline summary and stats
Final overview of what was processed and stored.  
This cell gives a quick sanity check before moving to Notebook 02.

In [ ]:
# Retrieve total count from ChromaDB collection
collection = chroma_client.get_collection(COLLECTION_NAME)
total_stored = collection.count()

print("=" * 60)
print("📊 PIPELINE SUMMARY")
print("=" * 60)
print(f"  Video URL:          {YOUTUBE_URL}")
print(f"  Video ID:           {video_id}")
print(f"  Transcript length:  {len(plain_text):,} characters")
print(f"  Chunks created:     {len(chunks)}")
print(f"  Chunks in ChromaDB: {total_stored}")
print(f"  Embedding model:    {EMBEDDING_MODEL}")
print(f"  ChromaDB path:      {CHROMA_PATH}")
print(f"  Collection name:    {COLLECTION_NAME}")
print("=" * 60)
print("✅ Pipeline complete — ready for Notebook 02: RAG Chain")

---

## ✅ What we built in this notebook

| Step | What | Why |
|------|------|-----|
| 1 | Extracted video ID from URL | Works with any YouTube URL format |
| 2 | Fetched transcript via `youtube-transcript-api` | Safe, fast — text only, no audio/video download |
| 3 | Added timestamps to each entry | Enables accurate source citation in answers |
| 4 | Split into 500-char chunks with 50-char overlap | Optimal chunk size for semantic search |
| 5 | Generated embeddings with `text-embedding-3-small` | Converts text to searchable vectors |
| 6 | Stored everything in ChromaDB | Persistent local vector database |
| 7 | Verified with a similarity search | Confirmed pipeline is working correctly |

## ➡️ Next: Notebook 02 — RAG Chain
In the next notebook we build the full question-answering chain on top of this knowledge base.